# Notebook 08 — AdaBoost 



## Step 0 — Setup

In [1]:
import os, warnings, pickle, copy, json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

warnings.filterwarnings("ignore")
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

# FIX: was "Algerian_Lamiaceae" (missing _v2) -- pointed at a different,
# possibly stale or nonexistent directory. Matches every other notebook now.
DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
OUTPUT_DIR   = os.path.join(DATASET_ROOT, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Output: /home/ali/Desktop/Algerian_Lamiaceae_v3/results


## Step 1 — Load Cached Features and Train/Val/Test Split

In [2]:
HC_CACHE    = os.path.join(OUTPUT_DIR, "hc_features_cache.npz")
DEEP_CACHE  = os.path.join(OUTPUT_DIR, "deep_features_cache.npz")
SPLIT_CACHE = os.path.join(OUTPUT_DIR, "train_test_split.npz")

X_hc = np.load(HC_CACHE, allow_pickle=True)["X"]
print(f"Handcrafted features: {X_hc.shape}")

if os.path.exists(DEEP_CACHE):
    X_deep = np.load(DEEP_CACHE, allow_pickle=True)["X"]
    X_all  = np.concatenate([X_deep, X_hc], axis=1)
    print(f"Deep features: {X_deep.shape}")
else:
    X_all = X_hc
    print("Deep features not found -- using handcrafted only")
print(f"Fused features: {X_all.shape}")

split = np.load(SPLIT_CACHE, allow_pickle=True)
X_train_sc = split["X_train_sc"]
X_val_sc   = split["X_val_sc"]
X_test_sc  = split["X_test_sc"]
y_train_sm = split["y_train_sm"]
y_val_raw  = split["y_val_raw"]
y_test_raw = split["y_test_raw"]
n_components = int(split["n_components"])

with open(os.path.join(OUTPUT_DIR, "label_encoder_species.pkl"), "rb") as f:
    le = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc1.pkl"), "rb") as f:
    sc1 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "sc2.pkl"), "rb") as f:
    sc2 = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "pca.pkl"), "rb") as f:
    pca = pickle.load(f)

class_names = list(le.classes_)
NUM_CLASSES = len(class_names)

print(f"Train: {X_train_sc.shape}  Val: {X_val_sc.shape}  Test: {X_test_sc.shape}")
print(f"Classes: {NUM_CLASSES} -- {class_names}")



Handcrafted features: (19145, 73)
Deep features: (19145, 1536)
Fused features: (19145, 1609)
Train: (35000, 300)  Val: (2372, 300)  Test: (2366, 300)
Classes: 7 -- [np.str_('Lamium_pooled'), np.str_('Lavandula_angustifolia'), np.str_('Lavandula_pooled'), np.str_('Leucas_aspera'), np.str_('Mentha_pooled'), np.str_('Ocimum_pooled'), np.str_('Plectranthus_pooled')]


## Step 2 — Evaluation Utilities (identical to previous classifiers)

In [3]:
N_BOOTSTRAP = 1000
SMALL_SUPPORT_THRESHOLD = 30

def evaluate(clf, Xtr, Xte, ytr, yte):
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    yproba = None
    auc_s = None
    if hasattr(clf, "predict_proba"):
        try:
            yproba = clf.predict_proba(Xte)
            auc_s = round(roc_auc_score(yte, yproba, multi_class="ovr",
                                         average="macro") * 100, 2)
        except Exception:
            pass
    return {
        "Accuracy":  round(accuracy_score(yte, yp) * 100, 2),
        "Precision": round(precision_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "Recall":    round(recall_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "F1":        round(f1_score(yte, yp, average="weighted", zero_division=0) * 100, 2),
        "AUC": auc_s,
        "_yp": yp, "_yproba": yproba, "_fitted_clf": clf,
    }

def bootstrap_ci(y_true, y_pred, y_proba=None, n_boot=N_BOOTSTRAP, seed=GLOBAL_SEED):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        accs.append(accuracy_score(yt, yp))
        precs.append(precision_score(yt, yp, average="weighted", zero_division=0))
        recs.append(recall_score(yt, yp, average="weighted", zero_division=0))
        f1s.append(f1_score(yt, yp, average="weighted", zero_division=0))
        if y_proba is not None:
            try:
                aucs.append(roc_auc_score(yt, y_proba[idx], multi_class="ovr", average="macro"))
            except ValueError:
                pass

    def summarize(arr):
        arr = np.array(arr) * 100
        return {"mean": round(float(arr.mean()), 2),
                "ci_lo": round(float(np.percentile(arr, 2.5)), 2),
                "ci_hi": round(float(np.percentile(arr, 97.5)), 2)}

    result = {"Accuracy": summarize(accs), "Precision": summarize(precs),
              "Recall": summarize(recs), "F1": summarize(f1s)}
    if y_proba is not None:
        result["AUC"] = summarize(aucs) if aucs else None
        result["AUC_valid_resamples"] = len(aucs)
        result["AUC_skipped_resamples"] = n_boot - len(aucs)
    return result

def per_class_report(y_true, y_pred, class_names, small_support_threshold=SMALL_SUPPORT_THRESHOLD):
    report = classification_report(y_true, y_pred, labels=list(range(len(class_names))),
                                    target_names=class_names, output_dict=True, zero_division=0)
    rows = []
    for cls in class_names:
        r = report[cls]
        rows.append({"Class": cls, "Precision": round(r["precision"] * 100, 2),
                     "Recall": round(r["recall"] * 100, 2),
                     "F1": round(r["f1-score"] * 100, 2), "Support": int(r["support"])})
    df = pd.DataFrame(rows)
    small = df[df["Support"] < small_support_threshold]
    if not small.empty:
        print(f"  \u26a0 Small test support (<{small_support_threshold} samples) -- "
              f"per-class scores below should not be over-interpreted (comment 7):")
        for _, r in small.iterrows():
            print(f"     {r['Class']:<28} support={r['Support']:>4}  "
                  f"P={r['Precision']:.1f}%  R={r['Recall']:.1f}%  F1={r['F1']:.1f}%")
    return df

def save_full_result(name, m, y_true, class_names):
    clf = m["_fitted_clf"]
    yp, yproba = m["_yp"], m["_yproba"]

    ci = bootstrap_ci(y_true, yp, yproba)
    per_class = per_class_report(y_true, yp, class_names)
    cm = confusion_matrix(y_true, yp, labels=list(range(len(class_names))))

    result = {k: v for k, v in m.items() if not k.startswith("_")}
    result["Classifier"] = name
    result["Bootstrap_CI_95"] = ci
    result["N_bootstrap"] = N_BOOTSTRAP
    result["PerClass"] = per_class.to_dict(orient="records")

    out_path = os.path.join(OUTPUT_DIR, f"result_{name.replace(' ','_')}.json")
    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    mdl_path = os.path.join(OUTPUT_DIR, f"model_{name.replace(' ','_')}.pkl")
    with open(mdl_path, "wb") as f:
        pickle.dump(clf, f)

    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{name.replace(' ','_')}.csv")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)

    auc_s = f"{m['AUC']}%" if m["AUC"] is not None else "n/a"
    print(f"  Acc={m['Accuracy']:6.2f}% [{ci['Accuracy']['ci_lo']:.1f}-{ci['Accuracy']['ci_hi']:.1f}]  "
          f"P={m['Precision']:6.2f}%  R={m['Recall']:6.2f}%  F1={m['F1']:6.2f}% "
          f"[{ci['F1']['ci_lo']:.1f}-{ci['F1']['ci_hi']:.1f}]  AUC={auc_s}")
    print(f"  Saved -> {out_path}")
    print(f"  Saved -> {mdl_path}")
    print(f"  Saved -> {cm_path}")

    with open(mdl_path, "rb") as f:
        reloaded = pickle.load(f)
    _ = reloaded.predict(X_test_sc[:1])
    print(f"  Smoke-test passed: reloaded model predicts without error.")

    return result

print("Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).")


Evaluation utilities ready (fit-and-return, bootstrap CI, per-class, confusion matrix).


## Step 3 — Baseline AdaBoost (n_estimators=100, matches original notebook)

In [4]:
from sklearn.ensemble import AdaBoostClassifier

name = "AdaBoost"
print(f"Training {name} ...")
print("AUC methodology: AdaBoostClassifier's native predict_proba (staged "
      "SAMME.R probabilities) used directly -- same reasoning as the other "
      "ensemble classifiers. AUC is One-vs-Rest, macro-averaged.")

clf = AdaBoostClassifier(n_estimators=100, random_state=GLOBAL_SEED)

m = evaluate(clf, X_train_sc, X_test_sc, y_train_sm, y_test_raw)
result_baseline = save_full_result(name, m, y_test_raw, class_names)


Training AdaBoost ...
AUC methodology: AdaBoostClassifier's native predict_proba (staged SAMME.R probabilities) used directly -- same reasoning as the other ensemble classifiers. AUC is One-vs-Rest, macro-averaged.
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=100.0%  F1=100.0%
  Acc= 61.16% [59.1-63.1]  P= 71.67%  R= 61.16%  F1= 63.98% [62.1-65.7]  AUC=90.32%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_AdaBoost.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_AdaBoost.pkl
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_AdaBoost.csv
  Smoke-test passed: reloaded model predicts without error.


## Step 4 — Optuna Hyperparameter Search 10

Tunes ensemble size, learning rate, and the base decision stump's depth
(AdaBoost's default base estimator is a depth-1 stump; allowing slightly
deeper stumps is a common and effective AdaBoost tuning axis). Tuned on
the held-out **validation** split; Test is not touched here.


In [5]:
import time
import optuna
from optuna.samplers import TPESampler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# --------------------------------------------------------------------------------------------
# WHY THIS STEP IS SLOW: AdaBoost trains its trees one after another (no multi-core option),
# and every depth-3 tree scans all 300 features of 12,000 samples. One trial = up to 150 trees
# -> 2-5 min per trial, x 40 trials = 1.5-3 h (longer if another notebook is using the CPU).
#
# The search is deterministic (TPE seed 42, same data): the run of 2026-10-19 on this exact
# v3 split already finished all 40 trials and found the parameters below; today's trial 0
# reproduced that run exactly (VAL F1 0.7714, same parameters). Re-running gives the same
# answer, so by default the finished result is re-used and only the final model is trained.
# Set RERUN_SEARCH = True to repeat the full search (e.g. if the split or features change).
# --------------------------------------------------------------------------------------------
RERUN_SEARCH       = False
SEARCH_SAMPLE_SIZE = 12000   # subsample only for the Optuna search
N_TRIALS           = 40
SEARCH_SPACE_DESC  = ("n_estimators ~ Int[20,150], learning_rate ~ LogUniform[1e-3,2.0], "
                      "base-tree max_depth ~ Int[1,3]; fitted on a 12,000-sample stratified "
                      "subset of the SMOTE-resampled training set")   # FIX: was undefined -> NameError
PREVIOUS_SEARCH = {"best_params": {"n_estimators": 118, "learning_rate": 0.32950253088857584,
                                   "stump_max_depth": 3},
                   "best_value": 0.8414, "date": "2026-10-19"}

if RERUN_SEARCH:
    X_search, _, y_search, _ = train_test_split(
        X_train_sc, y_train_sm, train_size=SEARCH_SAMPLE_SIZE,
        stratify=y_train_sm, random_state=GLOBAL_SEED)
    print(f"Optuna search on a {SEARCH_SAMPLE_SIZE:,}-sample stratified subset of the "
          f"{X_train_sc.shape[0]:,}-sample training set (final model retrained on the full set).")

    def objective_ada(trial):
        n_estimators = trial.suggest_int("n_estimators", 20, 150)
        learning_rate = trial.suggest_float("learning_rate", 1e-3, 2.0, log=True)
        stump_max_depth = trial.suggest_int("stump_max_depth", 1, 3)
        base = DecisionTreeClassifier(max_depth=stump_max_depth, random_state=GLOBAL_SEED)
        clf = AdaBoostClassifier(estimator=base, n_estimators=n_estimators,
                                 learning_rate=learning_rate, random_state=GLOBAL_SEED)
        clf.fit(X_search, y_search)
        return f1_score(y_val_raw, clf.predict(X_val_sc), average="weighted")

    t0 = time.time()
    study_ada = optuna.create_study(direction="maximize", sampler=TPESampler(seed=GLOBAL_SEED))
    study_ada.optimize(objective_ada, n_trials=N_TRIALS, show_progress_bar=True)
    best_params, best_value = study_ada.best_params, study_ada.best_value
    print(f"Search finished in {(time.time()-t0)/60:.1f} min")
else:
    best_params, best_value = PREVIOUS_SEARCH["best_params"], PREVIOUS_SEARCH["best_value"]
    print(f"Re-using the completed 40-trial search of {PREVIOUS_SEARCH['date']} "
          f"(set RERUN_SEARCH = True to repeat it).")

print(f"\nBest VAL F1(weighted) = {best_value*100:.2f}%")
print(f"Best params: {best_params}")

optuna_best_ada = {
    "Classifier": "AdaBoost_Optuna_BestTrial",
    "Val_F1_weighted": round(best_value * 100, 2),
    "BestParams": best_params,
    "NumTrials": N_TRIALS,
    "SearchSpace": SEARCH_SPACE_DESC,
    "SearchSampleSize": SEARCH_SAMPLE_SIZE,
    "Sampler": "TPESampler",
    "Seed": GLOBAL_SEED,
    "TuningSet": "validation (held out, not SMOTE-resampled)",
    "SearchRerunInThisSession": RERUN_SEARCH,
}
with open(os.path.join(OUTPUT_DIR, "optuna_result_AdaBoost.json"), "w") as f:
    json.dump(optuna_best_ada, f, indent=2)
print(optuna_best_ada)


Re-using the completed 40-trial search of 2026-10-19 (set RERUN_SEARCH = True to repeat it).

Best VAL F1(weighted) = 84.14%
Best params: {'n_estimators': 118, 'learning_rate': 0.32950253088857584, 'stump_max_depth': 3}
{'Classifier': 'AdaBoost_Optuna_BestTrial', 'Val_F1_weighted': 84.14, 'BestParams': {'n_estimators': 118, 'learning_rate': 0.32950253088857584, 'stump_max_depth': 3}, 'NumTrials': 40, 'SearchSpace': 'n_estimators ~ Int[20,150], learning_rate ~ LogUniform[1e-3,2.0], base-tree max_depth ~ Int[1,3]; fitted on a 12,000-sample stratified subset of the SMOTE-resampled training set', 'SearchSampleSize': 12000, 'Sampler': 'TPESampler', 'Seed': 42, 'TuningSet': 'validation (held out, not SMOTE-resampled)', 'SearchRerunInThisSession': False}


## Step 5 — Final Model: Retrain with Best Hyperparameters, Evaluate Once on Test

In [6]:
name = "AdaBoost_Optuna"
print(f"Training {name} (final retrained model, evaluated ONCE on held-out test) ...")

bp = best_params
t0 = time.time()
base_final = DecisionTreeClassifier(max_depth=bp["stump_max_depth"], random_state=GLOBAL_SEED)
clf_final = AdaBoostClassifier(estimator=base_final, n_estimators=bp["n_estimators"],
                                learning_rate=bp["learning_rate"], random_state=GLOBAL_SEED)

m_final = evaluate(clf_final, X_train_sc, X_test_sc, y_train_sm, y_test_raw)
print(f"  Final model fitted on {X_train_sc.shape[0]:,} samples in {(time.time()-t0)/60:.1f} min")
result_optuna = save_full_result(name, m_final, y_test_raw, class_names)

print("\n" + "="*70)
print("BASELINE (n_estimators=100, depth-1 stumps) vs. OPTUNA-TUNED -- compare on TEST:")
print("="*70)
print(f"  Baseline : Acc={result_baseline['Accuracy']:.2f}%  F1={result_baseline['F1']:.2f}%  "
      f"AUC={result_baseline['AUC']}%")
print(f"  Optuna   : Acc={result_optuna['Accuracy']:.2f}%  F1={result_optuna['F1']:.2f}%  "
      f"AUC={result_optuna['AUC']}%")


Training AdaBoost_Optuna (final retrained model, evaluated ONCE on held-out test) ...
  Final model fitted on 35,000 samples in 17.7 min
  ⚠ Small test support (<30 samples) -- per-class scores below should not be over-interpreted (comment 7):
     Leucas_aspera                support=  11  P=100.0%  R=100.0%  F1=100.0%
  Acc= 82.80% [81.3-84.3]  P= 84.57%  R= 82.80%  F1= 83.47% [82.0-84.9]  AUC=96.48%
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/result_AdaBoost_Optuna.json
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/model_AdaBoost_Optuna.pkl
  Saved -> /home/ali/Desktop/Algerian_Lamiaceae_v3/results/confusion_matrix_AdaBoost_Optuna.csv
  Smoke-test passed: reloaded model predicts without error.

BASELINE (n_estimators=100, depth-1 stumps) vs. OPTUNA-TUNED -- compare on TEST:
  Baseline : Acc=61.16%  F1=63.98%  AUC=90.32%
  Optuna   : Acc=82.80%  F1=83.47%  AUC=96.48%
